In this notebook, I extract words from the Anki deck and save them to my words collection.

## Words Extraction

In [1]:
import json

with open('deck.json', 'r') as f:
    data = json.load(f)

data[0]

{'deck': 'EnglishCustom',
 'card_id': 1745262268985,
 'note_id': None,
 'model': 'Basic',
 'front': 'вялый, заторможенный \nЯ чувствовал себя вялым после пробуждения \nDefinition: feeling weak, tired, or confused, especially after waking up or being ill',
 'back': 'groggy \nI felt groggy after waking up \n[sound:deck_test/audio_90.mp3]'}

In [5]:
def extract_word(data):
    back = data["back"]
    # first line, stripped
    return back.split("\n")[0].strip().lower()

extract_word(data[0])

'groggy'

In [6]:
words = list(map(extract_word, data))
words[:10]

['groggy',
 'temple',
 'spooky',
 'cunning',
 'hectic',
 'strife',
 'impertinent',
 'gastritis',
 'alarmist',
 'talk back']

In [7]:
import pandas as pd

df = pd.read_csv("words.csv")

df.head()

,word
0,a
1,a.m./A.M./am/AM
2,abandon
3,abandoned
4,ability


In [8]:
words_existing = df.word.tolist()

print("words existing:", len(words_existing))

words_new = list(map(extract_word, data))

print("words new:", len(words_new))

words_all = list(set(words_existing + words_new))

print("words all:", len(words_all))

words existing: 8657
words new: 887
words all: 9467


In [9]:
df_new = pd.DataFrame(words_all, columns=["word"])
df_new.to_csv("words.csv", index=False)

## New Words

In [33]:
import pandas as pd

existing_words = pd.read_csv("words.csv")
existing_words = set(existing_words.word.tolist())
print("existing words:", len(existing_words))

existing words: 9467


In [35]:
words = pd.read_csv("cefr_words.csv")
# lowercase all words
words['Base Word'] = words['Base Word'].str.lower()
print("words", len(words))

words 15696


In [36]:
words

,Base Word,Guideword,Level,Part of Speech,Topic
0,cattle,NaN,B1,NaN,animals
1,clothes,NaN,A1,NaN,clothes
2,albeit,NaN,C2,NaN,NaN
3,although,BUT,B1,NaN,communication
4,although,DESPITE,B1,NaN,communication
...,...,...,...,...,...
15691,with bated breath,IDIOM,C2,phrase,NaN
15692,be lost for words,IDIOM,C2,phrase,NaN
15693,(have) the best of both worlds,IDIOM,C1,phrase,NaN
15694,not be the end of the world,IDIOM,C2,phrase,NaN


In [37]:
# filter out words that are already in the existing words

words_filtered = words[~words['Base Word'].isin(existing_words)]
# level B1 and higher (not A1 or A2)
words_filtered = words_filtered[words_filtered['Level'].isin(['B2', 'C1', 'C2'])]
# only nouns, verbs, adjectives,
print("words filtered", len(words_filtered))

words filtered 4627


In [38]:
# group count by Part of Speech
words_filtered['Part of Speech'].value_counts()

Part of Speech
phrase          2924
phrasal verb     554
noun             501
adjective        402
verb             145
adverb            93
preposition        4
exclamation        1
Name: count, dtype: int64

In [39]:
len(words_filtered[words_filtered['Guideword'] == "IDIOM"])

483

In [40]:
# phrase, level B1, few samples
level = "C2"
print(len(words_filtered[(words_filtered['Part of Speech'] == "phrase") & (words_filtered['Level'] == level)]))
words_filtered[(words_filtered['Part of Speech'] == "phrase") & (words_filtered['Level'] == level)].sample(10)

1493


,Base Word,Guideword,Level,Part of Speech,Topic
13074,get you nowhere,NaN,C2,phrase,NaN
12177,not be cut out to be sth/not be cut out for sth,NaN,C2,phrase,people: personality
12468,hit the roof,IDIOM,C2,phrase,NaN
15521,at one time or another,NaN,C2,phrase,NaN
13947,to say nothing of sth,NaN,C2,phrase,NaN
12091,call it a day,IDIOM,C2,phrase,NaN
14485,be at a premium,NaN,C2,phrase,NaN
11396,your own flesh and blood,IDIOM,C2,phrase,NaN
12656,let your hair down,IDIOM,C2,phrase,NaN
12039,at sb's expense,NaN,C2,phrase,NaN


In [41]:
# rename columns
# words, guideword, pos, level, topic
words_filtered = words_filtered.rename(columns={
    'Base Word': 'word',
    'Guideword': 'guideword',
    'Part of Speech': 'pos',
    'Level': 'level',
    'Topic': 'topic'
})

In [42]:
words_filtered.to_csv("new_words.csv", index=False)